# Import

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
import random

# Pr Matrix

In [ ]:
## READ IN PRECURSOR MATRIX OUTPUT FROM DIA-NN
data = pd.read_csv("report.pr_matrix.tsv", sep="\t")
data.head()

,Protein.Group,Protein.Ids,Protein.Names,Genes,First.Protein.Description,Proteotypic,Stripped.Sequence,Modified.Sequence,Precursor.Charge,Precursor.Id,...,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A3.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A4.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A5.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A6.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A7.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A8.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A9.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A10.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A11.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A12.raw
0,Q96JP5;Q96JP5-2,Q96JP5;Q96JP5-2,ZFP91_HUMAN,ZFP91,NaN,1,AAAAAAAAAVSR,AAAAAAAAAVSR,2,AAAAAAAAAVSR2,...,NaN,NaN,NaN,NaN,NaN,NaN,18002.4,NaN,NaN,NaN
1,P36578,P36578,RL4_HUMAN,RPL4,NaN,1,AAAAAAALQAK,AAAAAAALQAK,2,AAAAAAALQAK2,...,1731690.00,2402730.00,2208060.00,2020480.0,1858000.0,1669390.0,2039340.0,2600800.0,2481350.0,2067210.00
2,Q8WUQ7;Q8WUQ7-2,Q8WUQ7;Q8WUQ7-2,CATIN_HUMAN,CACTIN,NaN,1,AAAAALSQQQSLQER,AAAAALSQQQSLQER,2,AAAAALSQQQSLQER2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4737.74
3,A6NIH7,A6NIH7,U119B_HUMAN,UNC119B,NaN,1,AAAAASAAGPGGLVAGK,AAAAASAAGPGGLVAGK,2,AAAAASAAGPGGLVAGK2,...,8331.42,3554.12,9025.23,20024.1,11283.8,NaN,NaN,NaN,NaN,7270.14
4,P52701,P52701;P52701-2;P52701-3,MSH6_HUMAN,MSH6,NaN,1,AAAAPGASPSPGGDAAWSEAGPGPRPLAR,AAAAPGASPSPGGDAAWSEAGPGPRPLAR,3,AAAAPGASPSPGGDAAWSEAGPGPRPLAR3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Read in IDs

In [ ]:
## READ IN SAMPLE IDENTIFIERS
ids = pd.read_excel("C:/Users/mjaconelli001/Documents/Identifiers.xlsx",sheet_name="Python")

## EXAMPLE: A549 RAB1

# Filter by A549
ids = ids[ids.Lysate=="A549"]
# then by Rab
ids = ids[(ids.Rab=="phos 1")|(ids.Rab=="Rab 1")]

ids["Rab"] = ids["Rab"].str.replace("phos ","phosRAB").str.replace("Rab ","RAB")
ids

,S-trap No.,Lysate,Rab
0,A1,A549,phosRAB1
1,A2,A549,phosRAB1
2,A3,A549,phosRAB1
3,A4,A549,phosRAB1
4,A5,A549,phosRAB1
5,A6,A549,phosRAB1
6,A7,A549,RAB1
7,A8,A549,RAB1
8,A9,A549,RAB1
9,A10,A549,RAB1


In [ ]:
# add sample identifiers as column names
data.columns = data.iloc[:,:10].columns.to_list() + ids["Rab"].to_list()
data.columns

Index(['Protein.Group', 'Protein.Ids', 'Protein.Names', 'Genes',
       'First.Protein.Description', 'Proteotypic', 'Stripped.Sequence',
       'Modified.Sequence', 'Precursor.Charge', 'Precursor.Id', 'phosRAB1',
       'phosRAB1', 'phosRAB1', 'phosRAB1', 'phosRAB1', 'phosRAB1', 'RAB1',
       'RAB1', 'RAB1', 'RAB1', 'RAB1', 'RAB1'],
      dtype='object')

## Filter out proteins with only 1 peptide identification

In [ ]:
## CHECK AND REMOVE PROTEIN(GROUP) IDENTIFIED BY SINGLE PEPTIDE
print(len(data),data["Protein.Group"].nunique())

dataf = data[data.duplicated("Protein.Group", keep=False) == True]
print(len(dataf),dataf["Protein.Group"].nunique()) ## FILTERED

33665 4412
32661 3408


In [ ]:
filt_pr = dataf["Protein.Group"].to_list() ## CREATE ID LIST OF DESIRED PG
filt_pr[0:10]

['Q96JP5;Q96JP5-2',
 'P36578',
 'Q8WUQ7;Q8WUQ7-2',
 'A6NIH7',
 'P52701',
 'A1X283',
 'Q8WWH5',
 'Q9UNF1;Q9UNF1-2',
 'Q01780;Q01780-2',
 'P02786']

# Protein Group Matrix

In [ ]:
## READ IN PG MATRIX OUTPUT FROM DIA-NN
pg = pd.read_csv("report.pg_matrix.tsv",sep="\t")
pg.head()

,Protein.Group,Protein.Names,Genes,First.Protein.Description,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A1.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A2.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A3.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A4.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A5.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A6.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A7.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A8.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A9.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A10.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A11.raw,Z:\ms-data\mrc-astral\2024\June\20240604\20240406_MJ_Eve_A12.raw
0,A0A075B6H7;P01624,KV315_HUMAN;KV37_HUMAN,IGKV3-15;IGKV3-7,NaN,1060060.0,804504.0,950082.0,625247.0,561450.0,500645.0,541316.0,728154.0,761728.0,1108690.0,885651.0,735432.0
1,A0A075B6K0;A0A075B6K4,LV310_HUMAN;LV316_HUMAN,IGLV3-10;IGLV3-16,NaN,56282800.0,38158200.0,49138500.0,37363600.0,42403500.0,75805000.0,78715200.0,121816000.0,98107400.0,71429300.0,70608500.0,48842000.0
2,A0A075B6R9,KVD24_HUMAN,IGKV2D-24,NaN,513571.0,327901.0,290336.0,242526.0,227992.0,232505.0,373758.0,223582.0,288236.0,584601.0,338972.0,305952.0
3,A0A075B6R9;A0A075B6S2;A0A0A0MRZ7;A2NJV5,KV229_HUMAN;KVD24_HUMAN;KVD26_HUMAN;KVD29_HUMAN,IGKV2-29;IGKV2D-24;IGKV2D-26;IGKV2D-29,NaN,NaN,17582300.0,16715700.0,11523900.0,NaN,10350900.0,15709900.0,NaN,NaN,NaN,NaN,NaN
4,A0A075B6S2;A0A0A0MRZ7;A2NJV5,KV229_HUMAN;KVD26_HUMAN;KVD29_HUMAN,IGKV2-29;IGKV2D-26;IGKV2D-29,NaN,26078900.0,18174600.0,17944000.0,9250990.0,11116600.0,10036200.0,19040300.0,18233500.0,17997000.0,25791500.0,16215600.0,11228700.0


In [ ]:
pgf = pg[pg["Protein.Group"].isin(filt_pr)] # FILTER DF TO RETAIN ONLY PG IDENTIFIED BY MORE THAN ONE PEPTIDE
len(pgf)

3408

In [ ]:
# FORMAT DF

pgf.drop(columns=["Protein.Names","First.Protein.Description"],inplace=True)
pgf = pgf.set_index("Genes")
pgf.columns = ["Protein.Group"] + data.iloc[:,10:].columns.to_list()
pgf.reset_index(inplace=True)
pgf.head()

C:\Users\mjaconelli001\AppData\Local\Temp\ipykernel_37076\2605490467.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pgf.drop(columns=["Protein.Names","First.Protein.Description"],inplace=True)


,Genes,Protein.Group,phosRAB1,phosRAB1,phosRAB1,phosRAB1,phosRAB1,phosRAB1,RAB1,RAB1,RAB1,RAB1,RAB1,RAB1
0,IGKV2-29;IGKV2D-26;IGKV2D-29,A0A075B6S2;A0A0A0MRZ7;A2NJV5,26078900.0,18174600.0,17944000.0,9250990.0,11116600.0,10036200.0,19040300.0,18233500.0,17997000.0,25791500.0,16215600.0,11228700.0
1,GATD3;GATD3B,A0A0B4J2D5;P0DPI2,NaN,NaN,NaN,97592.3,NaN,150353.0,96948.3,NaN,NaN,NaN,64500.4,71489.8
2,UBA6,A0AVT1,208187.0,278365.0,241293.0,299534.0,214484.0,347205.0,272879.0,240469.0,321589.0,285048.0,264272.0,277032.0
3,ESYT2,A0FGR8-2,317364.0,369262.0,319207.0,403168.0,380624.0,386309.0,531811.0,551379.0,603889.0,431774.0,515747.0,379890.0
4,MED19,A0JLT2,188926.0,98338.7,135808.0,109742.0,NaN,96882.3,NaN,NaN,102189.0,140450.0,NaN,NaN


## CV

# Filter by 100% Detection per Condition

In [ ]:
## SELECT ONLY COLUMNS CONTAINING INTENSITY VALUES FOR FILTERING
rabfilt = pgf.iloc[:,2:]
rabfilt.head()

,phosRAB1,phosRAB1,phosRAB1,phosRAB1,phosRAB1,phosRAB1,RAB1,RAB1,RAB1,RAB1,RAB1,RAB1
0,26078900.0,18174600.0,17944000.0,9250990.0,11116600.0,10036200.0,19040300.0,18233500.0,17997000.0,25791500.0,16215600.0,11228700.0
1,NaN,NaN,NaN,97592.3,NaN,150353.0,96948.3,NaN,NaN,NaN,64500.4,71489.8
2,208187.0,278365.0,241293.0,299534.0,214484.0,347205.0,272879.0,240469.0,321589.0,285048.0,264272.0,277032.0
3,317364.0,369262.0,319207.0,403168.0,380624.0,386309.0,531811.0,551379.0,603889.0,431774.0,515747.0,379890.0
4,188926.0,98338.7,135808.0,109742.0,NaN,96882.3,NaN,NaN,102189.0,140450.0,NaN,NaN


In [ ]:
## FILTER BY 100% DETECTION IN AT LEAST ONE SAMPLE CONDITION (I.E. 6/6 SAMPLES PER ONE CONDITION)

mstrind = []

for i in range(0, len(rabfilt.columns), 6):
    sliced = rabfilt.iloc[:,i:i+6].dropna(axis=0,how="any")
    mstrind += sliced.index.to_list()

In [36]:
ansind = set(mstrind) # merge by unique only

In [ ]:
rab_f = pgf[pgf.index.isin(ansind)] # COMPILE DF CONTAINING ONLY PG THAT WERE DETECTED 100% IN AT LEAST ONE SAMPLE CONDITION
rab_f

,Genes,Protein.Group,phosRAB1,phosRAB1,phosRAB1,phosRAB1,phosRAB1,phosRAB1,RAB1,RAB1,RAB1,RAB1,RAB1,RAB1
0,IGKV2-29;IGKV2D-26;IGKV2D-29,A0A075B6S2;A0A0A0MRZ7;A2NJV5,26078900.0,18174600.0,17944000.0,9250990.0,11116600.0,10036200.0,19040300.0,18233500.0,17997000.0,25791500.0,16215600.0,11228700.0
2,UBA6,A0AVT1,208187.0,278365.0,241293.0,299534.0,214484.0,347205.0,272879.0,240469.0,321589.0,285048.0,264272.0,277032.0
3,ESYT2,A0FGR8-2,317364.0,369262.0,319207.0,403168.0,380624.0,386309.0,531811.0,551379.0,603889.0,431774.0,515747.0,379890.0
5,SHTN1,A0MZ66;A0MZ66-3;A0MZ66-4,58430.3,57275.5,39531.1,114038.0,71314.7,93022.4,53274.0,60038.6,59356.3,67608.6,62665.0,65805.2
16,A2ML1,A8K2U0,186359.0,164704.0,125882.0,368323.0,128779.0,157940.0,207448.0,180110.0,156252.0,123839.0,124910.0,118013.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3396,DYNC1LI1,Q9Y6G9,169994.0,178320.0,96760.9,391299.0,247449.0,277954.0,187216.0,179995.0,200095.0,141918.0,277030.0,263275.0
3400,COX11,Q9Y6N1,NaN,73380.3,67704.9,58349.0,64957.3,85710.3,78428.4,84533.9,91341.3,86155.2,90792.0,67118.6
3401,SQOR,Q9Y6N5,723719.0,813630.0,862797.0,740528.0,725961.0,762617.0,986848.0,941425.0,920490.0,989958.0,872644.0,719004.0
3403,NUMBL,Q9Y6R0,4322740.0,4779360.0,4557110.0,2310360.0,3343060.0,3023360.0,4136780.0,4064330.0,3760240.0,5547790.0,NaN,3942690.0


In [ ]:
## WRITE OUT FILTERED PG MATRIX AS 'RAW' DATA FOR USE IN CURTAIN WEB TOOL

rab_f.to_csv("A549_RAB1_Raw.csv")

In [ ]:
######

## Log Transform

In [ ]:
## LOG2 TRANSFORM PG INTENSITIES
rab_f = rab_f.apply(lambda x: np.log2(x) if np.issubdtype(x.dtype, np.number) else x)
rab_f.head()

,Genes,Protein.Group,phosRAB1,phosRAB1,phosRAB1,phosRAB1,phosRAB1,phosRAB1,RAB1,RAB1,RAB1,RAB1,RAB1,RAB1
0,IGKV2-29;IGKV2D-26;IGKV2D-29,A0A075B6S2;A0A0A0MRZ7;A2NJV5,24.636380,24.115420,24.096998,23.141176,23.406212,23.258710,24.182553,24.120088,24.101253,24.620392,23.950879,23.420688
2,UBA6,A0AVT1,17.667520,18.086618,17.880427,18.192360,17.710511,18.405428,18.057902,17.875491,18.294859,18.120845,18.011664,18.079693
3,ESYT2,A0FGR8-2,18.275779,18.494285,18.284133,18.621022,18.538007,18.559396,19.020554,19.072685,19.203924,18.719917,18.976304,18.535222
5,SHTN1,A0MZ66;A0MZ66-3;A0MZ66-4,15.834429,15.805631,15.270700,16.799155,16.121912,16.505291,15.701144,15.873603,15.857114,16.044919,15.935372,16.005914
16,A2ML1,A8K2U0,17.507725,17.329516,16.941712,18.490612,16.974538,17.269017,17.662390,17.458519,17.253515,16.918106,16.930529,16.848586


# Imputation > MinProb

In [ ]:
# ADD REPLICATES TO COLUMN NAMES
renamer = defaultdict()

for column_name in rab_f.columns[rab_f.columns.duplicated(keep=False)].tolist():
    if column_name not in renamer:
        renamer[column_name] = [column_name+'.0']
    else:
        renamer[column_name].append(column_name +'.'+str(len(renamer[column_name])))
        
print(renamer)

# df

rab_f = rab_f.rename(
    columns=lambda column_name: renamer[column_name].pop(0)
    if column_name in renamer 
    else column_name)

rab_f.head()

defaultdict(None, {'phosRAB1': ['phosRAB1.0', 'phosRAB1.1', 'phosRAB1.2', 'phosRAB1.3', 'phosRAB1.4', 'phosRAB1.5'], 'RAB1': ['RAB1.0', 'RAB1.1', 'RAB1.2', 'RAB1.3', 'RAB1.4', 'RAB1.5']})


,Genes,Protein.Group,phosRAB1.0,phosRAB1.1,phosRAB1.2,phosRAB1.3,phosRAB1.4,phosRAB1.5,RAB1.0,RAB1.1,RAB1.2,RAB1.3,RAB1.4,RAB1.5
0,IGKV2-29;IGKV2D-26;IGKV2D-29,A0A075B6S2;A0A0A0MRZ7;A2NJV5,24.636380,24.115420,24.096998,23.141176,23.406212,23.258710,24.182553,24.120088,24.101253,24.620392,23.950879,23.420688
2,UBA6,A0AVT1,17.667520,18.086618,17.880427,18.192360,17.710511,18.405428,18.057902,17.875491,18.294859,18.120845,18.011664,18.079693
3,ESYT2,A0FGR8-2,18.275779,18.494285,18.284133,18.621022,18.538007,18.559396,19.020554,19.072685,19.203924,18.719917,18.976304,18.535222
5,SHTN1,A0MZ66;A0MZ66-3;A0MZ66-4,15.834429,15.805631,15.270700,16.799155,16.121912,16.505291,15.701144,15.873603,15.857114,16.044919,15.935372,16.005914
16,A2ML1,A8K2U0,17.507725,17.329516,16.941712,18.490612,16.974538,17.269017,17.662390,17.458519,17.253515,16.918106,16.930529,16.848586


In [ ]:
## IMPUTATION

# Calculate quantile == 0.01
q_prob = pd.DataFrame(data=rab_f.iloc[:,2:].quantile(q=0.01, axis=0, numeric_only=True, interpolation="nearest", method="single"))
q_prob = q_prob.T # transpose os wide format
q_prob_db = q_prob.values[0] # ensure treated as array as desired

# Calculate std
stds = (rab_f.iloc[:,2:].std(axis=1)) # by feature
std_med = stds.median() # median of all features

# Calculate Gaussian Distribution
n_sample = len(rab_f)
n_param = len(q_prob.columns)

qp_mat = pd.DataFrame(np.random.default_rng(seed=42).normal(q_prob_db, std_med, (n_sample, n_param)))
qp_mat

# Mask to orig df
cur_log2 = rab_f.replace(np.NaN,-1) # np.NaN == False

qp = cur_log2.iloc[:,2:] # for readability
qp_imp = qp.mask(qp.values == -1, qp_mat.values)
qp_imp = pd.concat([rab_f.iloc[:,:2], qp_imp],axis=1)
qp_imp

,Genes,Protein.Group,phosRAB1.0,phosRAB1.1,phosRAB1.2,phosRAB1.3,phosRAB1.4,phosRAB1.5,RAB1.0,RAB1.1,RAB1.2,RAB1.3,RAB1.4,RAB1.5
0,IGKV2-29;IGKV2D-26;IGKV2D-29,A0A075B6S2;A0A0A0MRZ7;A2NJV5,24.636380,24.115420,24.096998,23.141176,23.406212,23.258710,24.182553,24.120088,24.101253,24.620392,23.950879,23.420688
2,UBA6,A0AVT1,17.667520,18.086618,17.880427,18.192360,17.710511,18.405428,18.057902,17.875491,18.294859,18.120845,18.011664,18.079693
3,ESYT2,A0FGR8-2,18.275779,18.494285,18.284133,18.621022,18.538007,18.559396,19.020554,19.072685,19.203924,18.719917,18.976304,18.535222
5,SHTN1,A0MZ66;A0MZ66-3;A0MZ66-4,15.834429,15.805631,15.270700,16.799155,16.121912,16.505291,15.701144,15.873603,15.857114,16.044919,15.935372,16.005914
16,A2ML1,A8K2U0,17.507725,17.329516,16.941712,18.490612,16.974538,17.269017,17.662390,17.458519,17.253515,16.918106,16.930529,16.848586
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3396,DYNC1LI1,Q9Y6G9,17.375124,17.444109,16.562137,18.577912,17.916772,18.084487,17.514344,17.457597,17.610326,17.114698,18.079683,18.006211
3400,COX11,Q9Y6N1,13.805152,16.163105,16.046973,15.832420,15.987204,16.387181,16.259089,16.367242,16.478980,16.394650,16.470278,16.034425
3401,SQOR,Q9Y6N5,19.465070,19.634013,19.718662,19.498195,19.469533,19.540599,19.912468,19.844487,19.812043,19.917008,19.735034,19.455640
3403,NUMBL,Q9Y6R0,22.043515,22.188386,22.119688,21.139686,21.672738,21.527721,21.980077,21.954586,21.842393,22.403482,13.315595,21.910749


# PCA

# Differential Expression - BH

In [ ]:
## DIFFERENTIAL ANALYSIS USING LIMMA

# rpy2 - ebayes lmfit

import os
os.environ["R_HOME"] = r"C:\RSoftware\R-4.3.1"

import rpy2.robjects as ro
%load_ext rpy2.ipython

C:\Users\mjaconelli001\AppData\Local\anaconda3\Lib\site-packages\rpy2\robjects\packages.py:367: UserWarning: The symbol 'quartz' is not in this R namespace/package.
  warnings.warn(


In [59]:
from rpy2.robjects.packages import importr
limma = importr("limma")

In [60]:
from rpy2.robjects import pandas2ri # df conversion

In [61]:
# df conversion: rpy2

with (ro.default_converter + pandas2ri.converter).context():
  r_from_pd_df = ro.conversion.get_conversion().py2rpy(qp_imp.iloc[:,2:])

r_from_pd_df

phosRAB1.0,phosRAB1.1,phosRAB1.2,...,RAB1.3,RAB1.4,RAB1.5
24.636380,24.115420,24.096998,...,24.620392,23.950879,23.420688
17.667520,18.086618,17.880427,,18.120845,18.011664,18.079693
18.275779,18.494285,18.284133,,18.719917,18.976304,18.535222
15.834429,15.805631,15.270700,,16.044919,15.935372,16.005914
...,...,...,,...,...,...
13.805152,16.163105,16.046973,,16.394650,16.470278,16.034425
19.465070,19.634013,19.718662,,19.917008,19.735034,19.455640
22.043515,22.188386,22.119688,,22.403482,13.315595,21.910749
18.707106,19.843200,20.012322,,18.663530,18.111655,18.689275


In [62]:
%%R -i r_from_pd_df -o sample,condition

condition <- c()
sample <- c()
rename.cols <- c()
for (i in colnames(r_from_pd_df)) {
  x <- unlist(strsplit(i, "\\.")) # splits at . #> treats . as literal not regex
  sample <- cbind(sample, i) # sample has all data
  condition <- cbind(condition, paste(x[1], sep = ".")) # condition only want first 2 splits
}

In [63]:
%%R -i condition

condition

 [1] "phosRAB1" "phosRAB1" "phosRAB1" "phosRAB1" "phosRAB1" "phosRAB1"
 [7] "RAB1"     "RAB1"     "RAB1"     "RAB1"     "RAB1"     "RAB1"    


In [64]:
%%R -i sample

sample

 [1] "phosRAB1.0" "phosRAB1.1" "phosRAB1.2" "phosRAB1.3" "phosRAB1.4"
 [6] "phosRAB1.5" "RAB1.0"     "RAB1.1"     "RAB1.2"     "RAB1.3"    
[11] "RAB1.4"     "RAB1.5"    


In [65]:
%%R -o result

design <- model.matrix(~0 + condition)
colnames(design) <- gsub("condition", "", colnames(design))

fit <- lmFit(r_from_pd_df, design)
contmatrix <- makeContrasts(phosRAB1-RAB1, levels=design)
fit2 <- eBayes(contrasts.fit(fit, contmatrix))
result <- topTable(fit2, coef=1, number=Inf, sort.by = "none")

In [66]:
print(result) # be aware I've intentionally not reset index to preserve access to Protein.Group by index

         logFC    AveExpr         t   P.Value  adj.P.Val         B
0    -0.290159  23.920896 -1.042760  0.316981   0.776047 -6.921124
2    -0.082932  18.031943 -0.542464  0.597122   0.926205 -7.325833
3    -0.459331  18.691769 -3.190554  0.007490   0.127015 -3.484524
5     0.153175  15.979599  0.674000  0.512692   0.884313 -7.242174
16    0.240246  17.298731  0.921623  0.374324   0.813800 -7.039989
...        ...        ...       ...       ...        ...       ...
3396  0.029614  17.645283  0.097457  0.923923   0.997374 -7.477376
3400 -0.630438  16.018892 -1.719800  0.110331   0.569568 -6.054211
3401 -0.225101  19.666896 -1.837590  0.090215   0.532322 -5.874578
3403  1.214142  21.174885  0.921616  0.374328   0.813800 -7.039996
3404  0.333989  18.840984  1.039982  0.318220   0.776047 -6.923990

[2018 rows x 6 columns]


In [67]:
de = pd.DataFrame(result)
de["Comparison"] = "phosRAB1-RAB1"
de["Genes"] = qp_imp["Genes"].to_list()
de["Protein.Group"] = qp_imp["Protein.Group"].to_list()
de.head()

,logFC,AveExpr,t,P.Value,adj.P.Val,B,Comparison,Genes,Protein.Group
0,-0.290159,23.920896,-1.042760,0.316981,0.776047,-6.921124,phosRAB1-RAB1,IGKV2-29;IGKV2D-26;IGKV2D-29,A0A075B6S2;A0A0A0MRZ7;A2NJV5
2,-0.082932,18.031943,-0.542464,0.597122,0.926205,-7.325833,phosRAB1-RAB1,UBA6,A0AVT1
3,-0.459331,18.691769,-3.190554,0.007490,0.127015,-3.484524,phosRAB1-RAB1,ESYT2,A0FGR8-2
5,0.153175,15.979599,0.674000,0.512692,0.884313,-7.242174,phosRAB1-RAB1,SHTN1,A0MZ66;A0MZ66-3;A0MZ66-4
16,0.240246,17.298731,0.921623,0.374324,0.813800,-7.039989,phosRAB1-RAB1,A2ML1,A8K2U0


In [ ]:
## OUTPUT STATISTICS FOR USE IN CURTAIN WEB TOOL
de.to_csv("A549_RAB1_DE.csv")

In [ ]:
####